# Malaysian LRRK2 analysis - Association study

- **Project:** LRRK2 mutation spectrum and association study in a multi-ethnic cohort of Malaysian Parkinson’s Disease patients
- **Version:** Python/3.10.12
- **Created:** 05-NOVEMBER-2025
- **Last Update:** 12-DECEMBER-2025

## Description
1. Remove related individuals in the dataset and keep one of them
2. Run Association:
    - Unadjusted
    - Adjusted by sex, age and PCs
3. Combine the association results with the annotation

## Getting started

### Load python libraries

In [1]:
# Import necessary packages
import os
import pandas as pd
import numpy as np
from io import StringIO
from firecloud import api as fapi
from IPython.core.display import display, HTML
import urllib.parse
from google.cloud import bigquery
import sys as sys

# Define function
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

/tmp/ipykernel_3557/1504764740.py:21: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


### Install packages

In [4]:
%%capture
%%bash

# Install plink 1.9
cd /home/jupyter/
if test -e /home/jupyter/plink; then

echo "Plink is already installed in /home/jupyter/"
else
echo "Plink is not installed"
cd /home/jupyter

wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

unzip -o plink_linux_x86_64_20190304.zip
mv plink plink1.9

fi

In [5]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

unzip -o plink2_linux_x86_64_latest.zip

fi

In [6]:
%%bash

# chmod plink 1.9 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink1.9

In [7]:
%%bash

# chmod plink 2.0 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink2

## LRRK2 analysis in Malaysian samples

### Extract LRRK2 for Malaysian samples

In [29]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

# LRRK2: Specify gene boundaries in hg38 (from https://useast.ensembl.org/index.html)
/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rm \
--chr 12 \
--from-bp 40196744  \
--to-bp 40369285 \
--make-bed \
--snps-only just-acgt \
--out ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rm
  --chr 12
  --from-bp 40196744
  --make-bed
  --out MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2
  --snps-only just-acgt
  --to-bp 40369285

30088 MB RAM detected; reserving 15044 MB for main workspace.
358 out of 1945895 variants loaded from .bim file.
926 people (544 males, 382 females) loaded from .fam.
926 phenotype values loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 926 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate i

In [30]:
bim = pd.read_csv(f"{WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rm_LRRK2.bim", delim_whitespace = True, names = ["CHR", "RSID", "POS", "BP", "A1", "A2"])
bim["CHR"] = bim["CHR"].astype(str)
bim["BP"] = bim["BP"].astype(str)
bim["RSID"] = bim["CHR"].str.cat(bim["BP"], sep = "_")
bim["RSID"] = bim["RSID"].str.cat(bim["A2"], sep = "_")
bim["RSID"] = bim["RSID"].str.cat(bim["A1"], sep = "_")
bim.to_csv(f"{WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rm_LRRK2.bim", sep = "\t", header = False, index = False)

In [32]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

# Convert binary files into vcf file
/home/jupyter/plink2 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2 \
--rm-dup force-first \
--make-bed \
--out ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2
  --make-bed
  --out MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup
  --rm-dup force-first

Start time: Tue Nov 11 10:30:14 2025
30088 MiB RAM detected, ~27194 available; reserving 15044 MiB for main
workspace.
Using up to 8 compute threads.
926 samples (382 females, 544 males; 926 founders) loaded from
MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2.fam.
358 variants loaded from MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2.bim.
1 binary phenotype loaded (360 cases, 566 controls).
--rm-dup: 50 duplicated IDs, 66 variants removed.
Writing MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup.fam ... done.
Writing MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup.bim ... done.
Writing MALAY/GP2_merge_M

In [40]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup \
--extract ${label}/all_var.txt \
--recode A \
--out ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_allvar

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_allvar.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup
  --extract MALAY/all_var.txt
  --out MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_allvar
  --recode A

30088 MB RAM detected; reserving 15044 MB for main workspace.
292 variants loaded from .bim file.
926 people (544 males, 382 females) loaded from .fam.
926 phenotype values loaded from .fam.
--extract: 159 variants remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 926 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate 

In [41]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup \
--extract ${label}/all_var.txt \
--make-bed \
--out ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_allvar

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_allvar.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup
  --extract MALAY/all_var.txt
  --make-bed
  --out MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_allvar

30088 MB RAM detected; reserving 15044 MB for main workspace.
292 variants loaded from .bim file.
926 people (544 males, 382 females) loaded from .fam.
926 phenotype values loaded from .fam.
--extract: 159 variants remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 926 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate 

In [42]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_allvar \
--missing \
--out ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_allvar

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_allvar.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_allvar
  --missing
  --out MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_allvar

30088 MB RAM detected; reserving 15044 MB for main workspace.
159 variants loaded from .bim file.
926 people (544 males, 382 females) loaded from .fam.
926 phenotype values loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 926 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is 0.998438.
--missing: Sample missing data report written to
MAL

### Remove related individuals

In [56]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

# LRRK2: Specify gene boundaries in hg38 (from https://useast.ensembl.org/index.html)
/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2 \
--remove rm_rel.txt \
--make-bed \
--out ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2
  --make-bed
  --out MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2
  --remove rm_rel.txt

30088 MB RAM detected; reserving 15044 MB for main workspace.
358 variants loaded from .bim file.
926 people (544 males, 382 females) loaded from .fam.
926 phenotype values loaded from .fam.
--remove: 913 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 913 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate in remaining samples is 0

In [57]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

# Convert binary files into vcf file
/home/jupyter/plink2 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2 \
--rm-dup force-first \
--make-bed \
--out ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_nodup

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_nodup.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2
  --make-bed
  --out MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_nodup
  --rm-dup force-first

Start time: Tue Nov 11 10:36:00 2025
30088 MiB RAM detected, ~27153 available; reserving 15044 MiB for main
workspace.
Using up to 8 compute threads.
913 samples (376 females, 537 males; 913 founders) loaded from
MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2.fam.
358 variants loaded from MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2.bim.
1 binary phenotype loaded (355 cases, 558 controls).
--rm-dup: 50 duplicated IDs, 66 variants removed.
Writing MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_nodup.fam ... done.
Writing MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_nodup.bim ... done.
Writ

In [58]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_nodup \
--extract ${label}/all_var.txt \
--recode A \
--out ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_allvar

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_nodup
  --extract MALAY/all_var.txt
  --out MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar
  --recode A

30088 MB RAM detected; reserving 15044 MB for main workspace.
292 variants loaded from .bim file.
913 people (537 males, 376 females) loaded from .fam.
913 phenotype values loaded from .fam.
--extract: 159 variants remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 913 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyp

## LRRK2 association study

In [59]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_nodup \
--extract ${label}/all_var.txt \
--make-bed \
--out ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_allvar

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_nodup
  --extract MALAY/all_var.txt
  --make-bed
  --out MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar

30088 MB RAM detected; reserving 15044 MB for main workspace.
292 variants loaded from .bim file.
913 people (537 males, 376 females) loaded from .fam.
913 phenotype values loaded from .fam.
--extract: 159 variants remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 913 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyp

In [60]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_allvar \
--missing \
--out ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_allvar

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar
  --missing
  --out MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar

30088 MB RAM detected; reserving 15044 MB for main workspace.
159 variants loaded from .bim file.
913 people (537 males, 376 females) loaded from .fam.
913 phenotype values loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 913 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is 0.998636.
--missing: Sample missing data report writt

### Unadjusted

In [62]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

/home/jupyter/plink2 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_allvar \
--glm allow-no-covars firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
--pheno ${label}/cov_glm_${label}_age.txt \
--pheno-name PHENO \
--ci 0.95 \
--adjust \
--freq \
--out ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_allvar

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar.log.
Options in effect:
  --adjust
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar
  --ci 0.95
  --freq
  --glm allow-no-covars firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err
  --out MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar
  --pheno MALAY/cov_glm_MALAY_age.txt
  --pheno-name PHENO

Start time: Tue Nov 11 10:36:15 2025
30088 MiB RAM detected, ~27152 available; reserving 15044 MiB for main
workspace.
Using up to 8 compute threads.
913 samples (376 females, 537 males; 913 founders) loaded from
MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar.fam.
159 variants loaded from
MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar.bim.
1 binary phenotype loaded (355 cases, 558 controls).
Calculating all

### Adjusted

In [90]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

/home/jupyter/plink2 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_allvar \
--glm hide-covar firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
--pheno-name PHENO --covar-variance-standardize \
--pheno ${label}/cov_glm_${label}_age.txt \
--covar ${label}/cov_glm_${label}_age.txt \
--covar-name SEX,age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
--adjust \
--ci 0.95 \
--out ${label}/GP2_merge_${label}_qced_updated_rmrel_LRRK2_allvar_adj

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar_adj.log.
Options in effect:
  --adjust
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar
  --ci 0.95
  --covar MALAY/cov_glm_MALAY_age.txt
  --covar-name SEX,age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
  --covar-variance-standardize
  --glm hide-covar firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err
  --out MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar_adj
  --pheno MALAY/cov_glm_MALAY_age.txt
  --pheno-name PHENO

Start time: Tue Nov 11 11:58:12 2025
30088 MiB RAM detected, ~27003 available; reserving 15044 MiB for main
workspace.
Using up to 8 compute threads.
913 samples (376 females, 537 males; 913 founders) loaded from
MALAY/GP2_merge_MALAY_qced_updated_rmrel_LRRK2_allvar.fam.
159 variants loaded from


### Parse outputs for adjusted and unadjusted tests

In [ ]:
##############
# unadjusted #
##############

unadj_assoc         = pd.read_csv(f"{WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rmrel_LRRK2_allvar.PHENO.glm.logistic.hybrid", delim_whitespace = True)
unadj_assoc_correct = pd.read_csv(f"{WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rmrel_LRRK2_allvar.PHENO.glm.logistic.hybrid.adjusted", delim_whitespace = True)

unadj_assoc_red = unadj_assoc[~unadj_assoc["P"].isna()]
unadj_assoc_correct_red = unadj_assoc_correct[["ID", "UNADJ","GC","BONF","HOLM","SIDAK_SS","SIDAK_SD","FDR_BH","FDR_BY"]]

unadj_assoc_merge = pd.merge(unadj_assoc_red, unadj_assoc_correct_red,on = "ID", how = "left")

## Merging the OR and 95% CI
unadj_assoc_merge["OR 95%CI"] = (
    unadj_assoc_merge["OR"].astype(str) + 
    " (" + 
    unadj_assoc_merge["L95"].astype(str) + 
    " - " + 
    unadj_assoc_merge["U95"].astype(str) + 
    ")"
)
unadj_assoc_merge["OR 95%CI"]

## Keep them at 3 decimal places
unadj_assoc_merge["OR 95%CI"] = unadj_assoc_merge.apply(
    lambda row: f"{row['OR']:.3f} ({row['L95']:.3f} - {row['U95']:.3f})", axis=1
)

############
# adjusted #
############

adj_assoc         = pd.read_csv(f"{WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rmrel_LRRK2_allvar_adj.PHENO.glm.logistic.hybrid", delim_whitespace = True)
adj_assoc_correct = pd.read_csv(f"{WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rmrel_LRRK2_allvar_adj.PHENO.glm.logistic.hybrid.adjusted", delim_whitespace = True)

adj_assoc_red = adj_assoc[~adj_assoc["P"].isna()]
adj_assoc_correct_red = adj_assoc_correct[["ID", "UNADJ","GC","BONF","HOLM","SIDAK_SS","SIDAK_SD","FDR_BH","FDR_BY"]]

adj_assoc_merge = pd.merge(adj_assoc_red, adj_assoc_correct_red,on = "ID", how = "left")

## Merging the OR and 95% CI
adj_assoc_merge["OR 95%CI"] = (
    adj_assoc_merge["OR"].astype(str) + 
    " (" + 
    adj_assoc_merge["L95"].astype(str) + 
    " - " + 
    adj_assoc_merge["U95"].astype(str) + 
    ")"
)
adj_assoc_merge["OR 95%CI"]

## Keep them at 3 decimal places
adj_assoc_merge["OR 95%CI"] = adj_assoc_merge.apply(
    lambda row: f"{row['OR']:.3f} ({row['L95']:.3f} - {row['U95']:.3f})", axis=1
)

In [92]:
unadj_assoc_merge_red = unadj_assoc_merge[['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1_CT', 'ALLELE_CT', 'A1_CASE_CT', 'A1_CTRL_CT',
       'CASE_ALLELE_CT', 'CTRL_ALLELE_CT', 'A1_FREQ', 'A1_CASE_FREQ',
       'A1_CTRL_FREQ','P', 'OR 95%CI', 'GC', 'BONF']]


adj_assoc_merge_red = adj_assoc_merge[['ID', 'P', 'OR 95%CI', 'GC', 'BONF']]

assoc_rel_rm = pd.merge(unadj_assoc_merge_red, adj_assoc_merge_red, on = "ID", suffixes = ['_unadj', '_adj'])

In [93]:
# Keep the freq of missingness of each SNPs to be removed later
miss = pd.read_csv(f"{WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rmrel_LRRK2_allvar.lmiss", delim_whitespace = True)
miss_red = miss[["SNP", "F_MISS"]]
miss_red.rename(columns={"SNP":"ID"}, inplace = True)

/tmp/ipykernel_3557/2937702979.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  miss_red.rename(columns={"SNP":"ID"}, inplace = True)


In [94]:
# Add on annotation to the assoc results
assoc_merged_rel_rm = pd.merge(assoc_rel_rm, LRRK2_red, on = "ID", how = "left")

# Add on missingness results
assoc_merged_rel_rm_miss = pd.merge(assoc_merged_rel_rm, miss_red, on = "ID", how = "left")
assoc_merged_rel_rm_miss_cleaned = assoc_merged_rel_rm_miss.sort_values("F_MISS").drop_duplicates(subset=["POS", "REF", "ALT"], keep="last")

assoc_merged_rel_rm_miss_cleaned[assoc_merged_rel_rm_miss_cleaned["AA_change"] == "p.R1628R"]

,#CHROM,POS,ID,REF,ALT,A1_CT,ALLELE_CT,A1_CASE_CT,A1_CTRL_CT,CASE_ALLELE_CT,...,GC_adj,BONF_adj,Func.refGeneWithVer,Func.refGeneWithVer,Gene,NM_ID,exon,BP_change,AA_change,F_MISS
93,12,40320043,12_40320043_C_G,C,G,54,1822,27,27,708,...,0.172012,1,exonic,exonic,LRRK2,NM_198578.4,exon34,c.G4883G,p.R1628R,0.002191


In [95]:
assoc_merged_rel_rm_miss_cleaned.to_csv(f"{WORK_DIR}/{label}/UMKL_{label}_assoc_release11_rel_rm.txt", sep= "\t", header = True, index = False)


In [ ]:
# Define WORKSPACE_BUCKET elsewhere in the notebook according to the project specifications
shell_do(f"gsutil cp {WORK_DIR}/{label}/UMKL_{label}_assoc_release11_rel_keep.txt {WORKSPACE_BUCKET}/lrrk2_release11/assoc/")
shell_do(f"gsutil cp {WORK_DIR}/{label}/UMKL_{label}_assoc_release11_rel_rm.txt {WORKSPACE_BUCKET}/lrrk2_release11/assoc/")
